### RAG Pipeline - Data Ingestion to Vector DB

In [1]:
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/Users/aashitpaliwal/Desktop/RAGTEST/ragvenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## Read all pdf's inside directory
def processpdfs(pdf_directory):
  all_documents = []
  pdf_dir = Path(pdf_directory)

  pdf_files = list(pdf_dir.glob("**/*.pdf"))

  print(f"found {len(pdf_files)} PDF files to process")

  for pdf_file in pdf_files:
    print(f"\nProcessing: {pdf_file.name}")
    try:
      loader = PyMuPDFLoader(str(pdf_file))
      documents = loader.load()

      for doc in documents:
        doc.metadata['source_file'] = pdf_file.name
        doc.metadata['file_type'] = 'pdf'
      
      all_documents.extend(documents)
      print(f"Loaded {len(documents)} pages")
    
    except Exception as e:
      print(f"Error: {e}")
  
  print(f"\nTotal documents loaded: {len(all_documents)}")
  return all_documents

all_pdf_documents = processpdfs("../data")

found 11 PDF files to process

Processing: History 500 Week 7 Readings.pdf
Loaded 2 pages

Processing: History 500 Week 6 Readings.pdf
Loaded 2 pages

Processing: History 500 response 3.pdf
Loaded 2 pages

Processing: History 500 Week 10 Readings.pdf
Loaded 2 pages

Processing: History 500 Week 8 Readings.pdf
Loaded 2 pages

Processing: History 500 Week 5 Reading.pdf
Loaded 2 pages

Processing: History 500 Week 12 Readings.pdf
Loaded 2 pages

Processing: History 500 Week 9 Readings.pdf
Loaded 2 pages

Processing: History 500 Week 13 Readings.pdf
Loaded 2 pages

Processing: History 500 Week 14 Readings.pdf
Loaded 2 pages

Processing: History Week 11 Readings.pdf
Loaded 2 pages

Total documents loaded: 22


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m143 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '../data/pdf/History 500 Week 7 Readings.pdf', 'file_path': '../data/pdf/History 500 Week 7 Readings.pdf', 'total_pages': 2, 'format': 'PDF 1.4', 'title': 'History 500 Week 7 Readings', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': 'History 500 Week 7 Readings.pdf', 'file_type': 'pdf'}, page_content='In the readings Blake Scott and Krista Thompson both explore how the modern idea of the \nCaribbean as a tropical paradise was constructed through media. Scott focuses more on written \nworks while Thompson explores the effects photography has had. While they approach the \nsubject using different sources, both show that tourism did not simply emerge from the beauty of \nthe islands but was rather built from images and stories that turned colonial history into a \nmarketable fantasy. \nSc

In [4]:
## Text splitting into chunks

from sqlalchemy import text


def split_documents(documents, chunk_size=1000, chunk_overlap=200):
  """Split documents into smaller chunks for better rag performance"""
  text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
  )

  split_docs = text_splitter.split_documents(documents)
  print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

  # Show example of a chunk
  if split_docs:
    print(f"\nExample chunk:")
    print(f"Content: {split_docs[0].page_content[:200]}...")
    print(f"Metadata: {split_docs[0].metadata}")
  
  return split_docs


In [5]:
chunks=split_documents(all_pdf_documents)

Split 22 documents into 73 chunks

Example chunk:
Content: In the readings Blake Scott and Krista Thompson both explore how the modern idea of the 
Caribbean as a tropical paradise was constructed through media. Scott focuses more on written 
works while Thom...
Metadata: {'producer': 'Skia/PDF m143 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '../data/pdf/History 500 Week 7 Readings.pdf', 'file_path': '../data/pdf/History 500 Week 7 Readings.pdf', 'total_pages': 2, 'format': 'PDF 1.4', 'title': 'History 500 Week 7 Readings', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': 'History 500 Week 7 Readings.pdf', 'file_type': 'pdf'}


## Embeddings and Vector DB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
  def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
    """
    Initialize embedding manager

    Args:
      model_name: HuggingFace model name for sentence embeddings
    """
    self.model_name = model_name
    self.model = None
    self._load_model()
  
  def _load_model(self):
    """Load sentence transformer model"""
    try:
      print(f"Loading embedding model: {self.model_name}")
      self.model = SentenceTransformer(self.model_name)
      print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
    except Exception as e:
      print(f"Error in loading model {self.model_name}: {e}")
      raise
  
  def generate_embeddings(self, texts: List[str]) -> np.ndarray:
    """
    Generate embeddings for a list fo texts

    Args:
      texts: List of text strings to embed
    
    Returns:
      numpy array of embeddings with shape (len(texts), embedding_dim)
    """

    if not self.model:
      raise ValueError("Model not loaded")
    
    print(f"Generating embeddings for {len(texts)} texts ...")
    embeddings = self.model.encode(texts, show_progress_bar=True)
    print(f"Generated embeddings with shape {embeddings.shape}")
    return embeddings

## Initialize embedding manager

embedding_manager = EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8988.67it/s]


Model loaded successfully. Embedding dimension: 384


## Vector Store

In [8]:
from curses import meta


class VectorStore:

  def __init__ (self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
    """
    Initialize vector store

    Args:
      collection_name: Name of ChromaDB collection
      persist_directory: Directory to persist the Vector store
    """

    self.collection_name = collection_name
    self.persist_directory = persist_directory
    self.client = None
    self.collection = None
    self._initialize_store()
  
  def _initialize_store(self):
    """Initialize ChromaDb client and collection"""
    try:
      os.makedirs(self.persist_directory, exist_ok=True)
      self.client = chromadb.PersistentClient(path=self.persist_directory)

      self.collection = self.client.get_or_create_collection(
        name=self.collection_name,
        metadata={"description": "PDF document embeddinsg for RAG"}
      )

      print(f"Vector store initialized. Collection: {self.collection_name}")
      print(f"Existing documents in collection: {self.collection.count()}")

    except Exception as e:
      print(f"Error initializing vector store: {e}")
      raise
  
  def add_documents(self, documents: List[Any], embeddings: np.ndarray):
    """
    Add documents and their embeddings to the vector store
    
    Args:
      documents: List of langChain documents
      embeddings: Corresponding embeddings for the documents
    """

    if len(documents) != len(embeddings):
      raise ValueError("Num docs don't match num embeddings")
    
    print(f"Adding {len(documents)} documents to the vector store...")

    ids= []
    metadatas = []
    documents_text = []
    embeddings_list = []

    for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
      # Generate unique id
      doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
      ids.append(doc_id)

      # Prepare metadata
      metadata = dict(doc.metadata)
      metadata['doc_index'] = i
      metadata['content_length'] = len(doc.page_content)
      metadatas.append(metadata)

      # Document content
      documents_text.append(doc.page_content)

      # Embedding
      embeddings_list.append(embedding.tolist())
    
    # Add to collection
    try:
      self.collection.add(
        ids=ids,
        embeddings=embeddings_list,
        metadatas=metadatas,
        documents=documents_text
      )
      print(f"Successfully added {len(documents)} documents to vector store")
      print(f"Total documents in collection {self.collection.count()}")
    
    except Exception as e:
      print(f"Error addings documents to vector store: {e}")
      raise

vectorstore=VectorStore()
vectorstore


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 38


In [9]:
chunks

[Document(metadata={'producer': 'Skia/PDF m143 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '../data/pdf/History 500 Week 7 Readings.pdf', 'file_path': '../data/pdf/History 500 Week 7 Readings.pdf', 'total_pages': 2, 'format': 'PDF 1.4', 'title': 'History 500 Week 7 Readings', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': 'History 500 Week 7 Readings.pdf', 'file_type': 'pdf'}, page_content='In the readings Blake Scott and Krista Thompson both explore how the modern idea of the \nCaribbean as a tropical paradise was constructed through media. Scott focuses more on written \nworks while Thompson explores the effects photography has had. While they approach the \nsubject using different sources, both show that tourism did not simply emerge from the beauty of \nthe islands but was rather built from images and stories that turned colonial history into a \nmarketable fantasy. \nSc

In [10]:
# Convert text to embeddings 
texts = [doc.page_content for doc in chunks]

# Generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# Store in vector db
vectorstore.add_documents(chunks, embeddings)


Generating embeddings for 73 texts ...


Batches: 100%|██████████| 3/3 [00:01<00:00,  2.49it/s]

Generated embeddings with shape (73, 384)
Adding 73 documents to the vector store...
Successfully added 73 documents to vector store
Total documents in collection 111


## Retriever pipeline from VectorStore

In [11]:
from importlib import metadata
from turtle import distance


class RAGRetreiver:
  """Handles query-based retreival from the vector store"""

  def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
    """
    Initialize the retreiver

    Args:
      vector_store: Vector store with document embeddings
      embedding_manager: Manager for generating query embeddings
    """
    self.vector_store = vector_store
    self.embedding_manager = embedding_manager
  
  def retreive(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
    """
    Retreive relevant documents for a query

    Args:
      query: Search query
      top_k: Number of top results to return
      score_threshold: Minimum similarity score threshold
    
    Returns:
      List of dictionaries containing retreived documents and metadata
    """
    print(f"Retreiving documents for query: {query}")
    print(f"Top K: {top_k}, Score Threshold: {score_threshold}")

    # Generate query embedding 
    query_embedding = self.embedding_manager.generate_embeddings([query])[0]

    # Search vector store
    try:
      results = self.vector_store.collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
      )

      retreived_docs = []

      if results['documents'] and results['documents'][0]:
        documents = results['documents'][0]
        metadatas = results['metadatas'][0]
        distances = results['distances'][0]
        ids = results['ids'][0]

        for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
          similarity_score = 1 - distance

          if similarity_score >= score_threshold:
            retreived_docs.append({
              'id': doc_id,
              'content': document,
              'metadata': metadata,
              'similarity_score': similarity_score,
              'distance': distance,
              'rank': i + 1
            })
        
        print(f"Retreived {len(retreived_docs)} documents (after filtering)")
      else:
        print(f"No documents found")
      
      return retreived_docs
    
    except Exception as e:
      print(f"Error during retreival: {e}")
      return []

rag_retreiver = RAGRetreiver(vectorstore, embedding_manager)

In [12]:
rag_retreiver

In [15]:
rag_retreiver.retreive("Explain Cuban Revolution", )

Retreiving documents for query: Explain Cuban Revolution
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.77it/s]

Generated embeddings with shape (1, 384)
Retreived 5 documents (after filtering)


[{'id': 'doc_66e0ff3f_5',
  'content': "tourism turned Cuba into a stage where the meanings of socialism and global belonging were \nconstantly changing. The sparkle of new hotels promised progress and recovery, yet they \ndepended on narratives that celebrated the island’s charm while masking the economic issues \nand social tensions underneath.  \nUltimately, these texts show that tourism in Cuba functions as a form of global storytelling that \nshapes how the island is seen and who gets to participate in that image. While it offers the hope \nof economic relief, it often reproduces the same inequalities and silences that the revolution \nonce sought to dismantle. Thus, leaving Cuba not as a show of socialism's triumphs but rather a \ndemonstration of western and capitalist leverage upon the island.",
  'metadata': {'creator': '',
   'producer': 'Skia/PDF m144 Google Docs Renderer',
   'source_file': 'History 500 Week 12 Readings.pdf',
   'creationdate': '',
   'format': 'PDF 1.4',
 